# Eigenes Experiment: Wählen Sie eine Frage
**Internet of Production – KI, Woche 2, Block B** · 35 min · Zweiergruppen

Die Pipeline aus Übung 2 ist vorbereitet. Wählen Sie **eine** von drei Fragen, ändern Sie **eine Sache** und füllen Sie am Ende das Protokoll aus.

| Option | Frage | Abschnitt |
|---|---|---|
| 1 | Baum und Schwellwert: Welche Tiefe, Blattgröße und welcher Schwellwert passen zu einer 10-Minuten-Prüfung und einem 12-Stunden-Ausfall? | 1 |
| 2 | Mehr Merkmale: Hilft ein weiteres Merkmal, und gibt es einen physikalischen Grund? | 2 |
| 3 | Validierung über die Zeit: Wie stabil ist das Ergebnis? | 3 |

**Regeln:** Hypothese aufschreiben, *bevor* Sie eine Zelle ausführen · genau eine Sache auf einmal ändern · nur Aufteilung nach Zeit (die ersten 70 % Training, die letzten 30 % Test) · Zahl der Ereignisse angeben, nicht nur Prozent.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score, confusion_matrix

DATA = Path('../../00_Data/printfarm')
HORIZON = 360                       # minutes: warn 6 hours ahead
FEATURES = ['E30', 'E120', 'slope']

def make_blockage_data(k, horizon=HORIZON):
    """The pipeline of exercise 2 of week 1 in one function: one row per 10th printing minute."""
    d = pd.read_csv(DATA / f'Printer_{k}.csv', sep=';', decimal=',', low_memory=False).dropna(how='all')
    d['Timestamp'] = pd.to_datetime(d['Timestamp'], format='%d.%m.%Y %H:%M')
    d = d.set_index('Timestamp')
    d['Energy'] = d['Total_Power'].diff()
    e = pd.read_csv(DATA / f'Printer_{k}_error.csv', sep=';').dropna(how='all')
    e['Timestamp'] = pd.to_datetime(e['Timestamp'], format='%d.%m.%Y %H:%M')
    blockages = np.sort(e.loc[e['Error'] == 'Extruder blocked', 'Timestamp'].values)
    p = d[d['State'] == 'printing'].copy()
    p['E'] = p['Energy'] * 1000                              # Wh per minute
    idx = p.index.values
    nxt = np.searchsorted(blockages, idx)
    mtb = np.where(nxt < len(blockages), (blockages[np.minimum(nxt, len(blockages) - 1)] - idx) / np.timedelta64(1, 'm'), np.inf)
    p['y'] = ((mtb > 0) & (mtb <= horizon)).astype(int)
    p['E30'] = p['E'].rolling(30).mean()
    p['E120'] = p['E'].rolling(120).mean()
    p['slope'] = p['E120'] - p['E120'].shift(240)
    p['Temp_Extruder_m'] = p['Temp_Extruder'].rolling(120).mean()
    p['Temp_Bed_m'] = p['Temp_Bed'].rolling(120).mean()
    return p.dropna(subset=FEATURES + ['Temp_Extruder_m']).iloc[::10], pd.to_datetime(blockages)

def report(name, y_true, y_pred, score=None):
    auc = f'   AUC {roc_auc_score(y_true, score):5.3f}' if score is not None else ''
    print(f'{name:36s} accuracy {accuracy_score(y_true, y_pred):6.1%}   recall {recall_score(y_true, y_pred, zero_division=0):6.1%}   precision {precision_score(y_true, y_pred, zero_division=0):6.1%}{auc}')

data, blockages = make_blockage_data(1)
cut = int(len(data) * 0.7)                      # split by time: first 70 % train, last 30 % test
train, test = data.iloc[:cut], data.iloc[cut:]
X_train, y_train, X_test, y_test = train[FEATURES], train['y'], test[FEATURES], test['y']
print(len(data), 'samples,', f"{data['y'].mean():.1%} class 1; train {len(train)}, test {len(test)}")

results = []          # your experiment log; every run adds a row
def log(name, y_true, score, threshold=0.5):
    """Evaluate a score on the test set and add one row to the log."""
    pred = (score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    results.append({'run': name, 'events (class 1 samples)': int(tp + fn), 'recall': tp / (tp + fn),
                    'precision': tp / (tp + fp) if tp + fp else float('nan'), 'false alarms': int(fp),
                    'AUC': roc_auc_score(y_true, score)})
    display(pd.DataFrame(results).set_index('run').round(3))

tree0 = DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight='balanced', random_state=0).fit(X_train, y_train)
log('reference: tree depth 3, leaf 200', y_test, tree0.predict_proba(X_test)[:, 1])
print('blockages in the test period:', int(((blockages >= test.index[0]) & (blockages <= test.index[-1])).sum()))

## Protokoll (laufend ausfüllen)
**Option / Hypothese** (vor dem Lauf): *Ihre Antwort*

## Abschnitt 1: Baum und Schwellwert
Ändern Sie **eine** der Größen `MAX_DEPTH`, `MIN_LEAF`, `THRESHOLD`, `CLASS_WEIGHT`, führen Sie aus und vergleichen Sie die neue Zeile mit der Referenz. Ein Fehlalarm ist hier eine Stichproben-Minute (jede 10. Druckminute); eine Prüfung dauert 10 Minuten, eine Blockade kostet einen Auftrag und etwa 12 Stunden.

**Hinweis:** (1) Überanpassung zeigt sich nur bei kleinem `MIN_LEAF` (probieren Sie 1) und großem `MAX_DEPTH` (6 und mehr); `MIN_LEAF = 200` verhindert sie weitgehend. (2) `CLASS_WEIGHT = 'balanced'` verschiebt den Betriebspunkt ähnlich wie ein niedrigerer `THRESHOLD` (mehr Recall, weniger Precision) und ändert, welche Splits gewählt werden. Beurteilen Sie es deshalb mit der AUC und dem Recall bei ähnlicher Precision, nicht nur mit dem Recall.

In [ ]:
MAX_DEPTH, MIN_LEAF, THRESHOLD, CLASS_WEIGHT = 3, 200, 0.5, 'balanced'     # change ONE of them per run (CLASS_WEIGHT: 'balanced' or None)
m = DecisionTreeClassifier(max_depth=MAX_DEPTH, min_samples_leaf=MIN_LEAF, class_weight=CLASS_WEIGHT, random_state=0).fit(X_train, y_train)
log(f'tree depth {MAX_DEPTH}, leaf {MIN_LEAF}, weight {CLASS_WEIGHT}, threshold {THRESHOLD}', y_test, m.predict_proba(X_test)[:, 1], THRESHOLD)
print(export_text(m, feature_names=FEATURES, decimals=3, max_depth=3))

## Abschnitt 2: Mehr Merkmale
Kandidaten: `Temp_Extruder_m`, `Temp_Bed_m` (Mittel der letzten 2 Stunden). Vergleichen Sie **nur Energie** mit **Energie + zusätzlichem Merkmal** in fünf Zeitblöcken und achten Sie auf die Zahl der Blockaden je Block.

In [ ]:
EXTRA = ['Temp_Bed_m']                              # change: e.g. ['Temp_Extruder_m'] or both

def over_time(features, make=lambda: DecisionTreeClassifier(max_depth=3, min_samples_leaf=200, class_weight='balanced', random_state=0), n_splits=5):
    rows = []
    for i, (a, bb) in enumerate(TimeSeriesSplit(n_splits=n_splits).split(data), 1):
        tr, te = data.iloc[a], data.iloc[bb]
        mdl = make().fit(tr[features], tr['y'])
        rows.append({'block': i, 'blockages in block': int(((blockages >= te.index[0]) & (blockages <= te.index[-1])).sum()),
                     'recall': recall_score(te['y'], mdl.predict(te[features])),
                     'precision': precision_score(te['y'], mdl.predict(te[features]), zero_division=0),
                     'AUC': roc_auc_score(te['y'], mdl.predict_proba(te[features])[:, 1])})
    return pd.DataFrame(rows).set_index('block')

base_f, ext_f = over_time(FEATURES), over_time(FEATURES + EXTRA)
cmp = pd.DataFrame({'blockages': base_f['blockages in block'], 'AUC energy only': base_f['AUC'], 'AUC + extra': ext_f['AUC']})
cmp['difference'] = cmp['AUC + extra'] - cmp['AUC energy only']
display(cmp.round(3)); print('mean difference', cmp['difference'].mean().round(3), ', better in', int((cmp['difference'] > 0).sum()), 'of', len(cmp), 'blocks')

dd = data.copy(); dd['group'] = np.where(dd['y'] == 1, 'within 6 h before blockage', 'other')
display(dd.groupby('group')[['E120'] + EXTRA].mean().round(3))

## Abschnitt 3: Validierung über die Zeit
Vergleichen Sie für ein Modell Ihrer Wahl die zufällige Aufteilung mit der Aufteilung nach Zeit und schieben Sie dann die Aufteilung durch die Zeit (`N_SPLITS` Blöcke).

In [ ]:
MODEL = lambda: DecisionTreeClassifier(random_state=0)       # try: 1-NN, KNN k = 25, tree depth 3 ...
N_SPLITS = 5

m_time = MODEL().fit(X_train, y_train)
Xr_tr, Xr_te, yr_tr, yr_te = train_test_split(data[FEATURES], data['y'], test_size=0.3, random_state=0, stratify=data['y'])
m_rand = MODEL().fit(Xr_tr, yr_tr)
print(f"recall, time split   {recall_score(y_test, m_time.predict(X_test)):.2f}")
print(f"recall, random split {recall_score(yr_te, m_rand.predict(Xr_te)):.2f}")

blocks = over_time(FEATURES, make=MODEL, n_splits=N_SPLITS)
display(blocks.round(3))
print('recall per block: min', round(blocks['recall'].min(), 2), ' max', round(blocks['recall'].max(), 2), ' blockages per block', blocks['blockages in block'].tolist())

## Protokoll und Ergebnis
Übertragen Sie das in Ihre Pitch-Folie (eine je Gruppe).

| | |
|---|---|
| **Hypothese** | |
| **Experiment** (eine Änderung, Daten, Split, Kennzahl) | |
| **Ergebnis** (Zahlen mit Zahl der Ereignisse) | |
| **Schlussfolgerung** (bestätigt / verworfen / offen; nächster Schritt; was bleibt unsicher) | |